# 03 — Per-capita crime rates by district

Same question as London notebook 03: does population explain the spread in
raw crime counts, or does the ranking change once we account for it?

In [ ]:
import sys
sys.path.append("../../src")

import matplotlib.pyplot as plt
import seaborn as sns

from load_data import load_force_data, load_population
from clean import clean_crime_data, add_area_column, WEST_MERCIA_DISTRICTS

sns.set_theme(style="whitegrid")

wm = load_force_data("west-mercia")
wm = clean_crime_data(wm)
wm = add_area_column(wm, column_name="District")
wmd = wm[wm["District"].isin(WEST_MERCIA_DISTRICTS)]

# load_population is the same generic loader London's notebook 03 uses --
# just called with West Mercia's district set and "District" as the
# column name instead of "Borough".
pop = load_population(WEST_MERCIA_DISTRICTS, name_column="District")
pop.sort_values("Population", ascending=False)

### A naming mismatch we had to fix in `src/load_data.py`

The first version of this merge silently produced a missing population for
Herefordshire. Cause: ONS's official name for it is **"Herefordshire,
County of"** — a ceremonial-county naming quirk — while the crime data's
LSOA names just say "Herefordshire". `src/clean.py` now has a
`NAME_ALIASES` dict that every loader applies, so this is already handled
before you get here — but it's a good example of why checking `set`
equality before trusting a merge (below) matters.

In [ ]:
set(pop["District"]) == WEST_MERCIA_DISTRICTS

In [ ]:
counts = wmd.groupby("District").size().rename("Crimes").reset_index()
merged = counts.merge(pop, on="District", validate="one_to_one")
merged["rate_per_1000"] = merged["Crimes"] / merged["Population"] * 1000
merged.sort_values("rate_per_1000", ascending=False)

## Chart — crime rate per capita by district

In [ ]:
rate_sorted = merged.sort_values("rate_per_1000")

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(rate_sorted["District"], rate_sorted["rate_per_1000"], color="#55A868")
ax.set_xlabel("Recorded crimes per 1,000 residents (annual)")
ax.set_title("Crime rate per capita by West Mercia district")
fig.tight_layout()

### Findings

- **Worcester jumps to #1 per-capita** (~121 per 1,000), despite being only
  the 4th-largest district by population — a similar *shape* of finding to
  Westminster in London (crime higher than population alone predicts), but
  nowhere near as extreme (121 vs. Westminster's ~465). Worcester is the
  county town of Worcestershire — its city centre likely draws non-resident
  shoppers/visitors the way Westminster does, just at a much smaller scale.
- **Shropshire and Telford and Wrekin drop out of the top spots**
  per-capita, despite leading on raw counts — the same reordering effect
  we saw in London (Camden, City of London) once population is accounted
  for.
- Unlike London, there's **no extreme outlier** on the scale of
  Westminster/City of London here — worth keeping in mind for notebook 04:
  we may not need to exclude anything before testing against deprivation.

## Chart — population vs. total crime

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(merged["Population"], merged["Crimes"], color="#4C72B0", s=45)
for _, row in merged.iterrows():
    ax.annotate(
        row["District"], (row["Population"], row["Crimes"]),
        textcoords="offset points", xytext=(6, 4), fontsize=8,
    )
ax.set_xlabel("Resident population (mid-2024)")
ax.set_ylabel("Recorded crimes (May 2025 - May 2026)")
ax.set_title("Population vs. total recorded crime, West Mercia districts")
fig.tight_layout()

With only 9 districts, every point is labelled (not just outliers) — the
"selective labels" rule from London's notebooks was about not cluttering
33 points; with 9, labelling all of them is still readable.